In [ ]:
# import package
import os

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import time
lib_dir = '/content/drive/MyDrive/Project/PlasticityDecoding'
os.chdir(lib_dir)
from modules.snn import *

dtype = torch.float64


In [ ]:
# Check whether a GPU is available
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [ ]:
MNIST_data = np.load('/content/drive/MyDrive/Project/PlasticityDecoding/data/MNIST_data_500.npy',allow_pickle=True)
x_data = torch.tensor(MNIST_data.item().get('spk'), dtype=float, device=device, requires_grad=False)
y_data = torch.tensor(MNIST_data.item().get('type'), dtype=torch.long, device=device, requires_grad=False)
del MNIST_data

SRNN_args = {
    'nb_inputs':784,
    'nb_hidden':225,
    'nb_outputs':10,
    'tau_mem':500e-3, #10e-3
    'tau_syn':5e-3,
    'time_step':1e-3,
    'nb_steps':500,
    'device':device,
    'batch_size':300,
    'w_max':5e-3,
    'w_min':0,
    'x_tar':0.70,
    'act_noise':0.05,
    'weight_noise':5e-4,
    'bf_b':0.4,
    'burst_thres':4,
    'err2bf_scaling':0.4 * 0.1,
}


In [ ]:
save_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/data/FixedRepresentation01'
repeat_ind = 0
trace_num = 600
trace_num0 = 0
learning_epochs = 30
while trace_num0 < trace_num:
    w_STDP = []
    snn_STDP = SRNN(SRNN_args).to(device)
    torch.save(snn_STDP,save_dir + '/snn_STDP.pth')
    spk_rec_e, x_pre_rec = snn_STDP(x_data)

    # Set firing rate baseline as the initial firing rate
    if torch.rand(1) > 0.5:
        fr_baseline = 3
    else:
        fr_baseline = 4
    # print(fr_baseline)
    # if fr_baseline < 3:
    #     continue

    Spk_STDP_trace = []
    Spk_STDP_tsp = []
    STDP_lr = 4e-1
    for epoch in range(0,learning_epochs):
        spk_rec_e, x_pre_rec = snn_STDP(x_data)
        snn_STDP = STDP(snn_STDP, SRNN_args, STDP_lr, spk_rec_e, x_pre_rec)
        spk_rec_e1 = event_2_burst(spk_rec_e, SRNN_args)
        Spk_STDP_tsp.append(torch.argwhere(spk_rec_e1 > 0))
        w_STDP.append(snn_STDP.state_dict()['w1'].clone())
    fr_STDP = spk_rec_e1.detach().sum(dim=1)

    if spk_rec_e1.detach().sum(dim=1).max() > 14:
        continue

    # event_rec_e = Spk2event(spk_rec_e, SRNN_args['burst_thres'])
    er_tar = spk_rec_e.detach().sum(dim=1)

    snn_BDP = SRNN(SRNN_args).to(device)
    snn_BDP = torch.load(save_dir + '/snn_STDP.pth')
    Spk_BDP_trace = []
    Spk_BDP_tsp = []
    BDP_lr = 3e-2
    for epoch in range(0,learning_epochs):
        spk_rec_e, x_pre_rec = snn_BDP(x_data)
        snn_BDP, spk_rec_e1 = BDP(snn_BDP, x_pre_rec, spk_rec_e, er_tar, SRNN_args, BDP_lr)
        Spk_BDP_tsp.append(torch.argwhere(spk_rec_e1 > 0))
    fr_BDP = spk_rec_e1.detach().sum(dim=1)

    snn_BP1 = SRNN(SRNN_args).to(device)
    snn_BP1 = torch.load(save_dir + '/snn_STDP.pth')
    BP_lr = 3e-4
    optimizer = torch.optim.Adam(snn_BP1.parameters(), lr=BP_lr)
    Loss_fn_BP1 = nn.SmoothL1Loss(reduction='mean')
    Spk_BP1_trace = []
    Spk_BP1_tsp = []
    for epoch in range(0,learning_epochs):
        spk_rec_e, x_pre_rec = snn_BP1(x_data)
        # event_rec_e = Spk2event(spk_rec_e, SRNN_args['burst_thres'])
        Loss_BP1 = Loss_fn_BP1(spk_rec_e.sum(dim=1), er_tar)
        optimizer.zero_grad()
        Loss_BP1.backward()
        optimizer.step()
        snn_BP1 = BP_fluc(snn_BP1, SRNN_args, BP_lr)
        spk_rec_e1 = event_2_burst_BP(spk_rec_e, SRNN_args)
        Spk_BP1_tsp.append(torch.argwhere(spk_rec_e1 > 0))
    fr_BP1 = spk_rec_e1.detach().sum(dim=1)

    snn_BP2 = SRNN(SRNN_args).to(device)
    snn_BP2 = torch.load(save_dir + '/snn_STDP.pth')
    BP_lr = 3e-4
    optimizer = torch.optim.Adam(snn_BP2.parameters(), lr=BP_lr)
    Loss_fn_BP2 = nn.MSELoss(reduction='mean')
    Spk_BP2_trace = []
    Spk_BP2_tsp = []
    for epoch in range(0,learning_epochs):
        spk_rec_e, x_pre_rec = snn_BP2(x_data)
        # event_rec_e = Spk2event(spk_rec_e, SRNN_args['burst_thres'])
        Loss_BP2 = Loss_fn_BP2(spk_rec_e.sum(dim=1), er_tar)
        optimizer.zero_grad()
        Loss_BP2.backward()
        optimizer.step()
        snn_BP2 = BP_fluc(snn_BP2, SRNN_args, BP_lr)
        spk_rec_e1 = event_2_burst_BP(spk_rec_e, SRNN_args)
        Spk_BP2_tsp.append(torch.argwhere(spk_rec_e1 > 0))
    fr_BP2 = spk_rec_e1.detach().sum(dim=1)

    snn_NC = SRNN(SRNN_args).to(device)
    snn_NC = torch.load(save_dir + '/snn_STDP.pth')
    Spk_NC_trace = []
    Spk_NC_tsp = []
    for epoch in range(0,learning_epochs):
        spk_rec_e, x_pre_rec = snn_NC(x_data)
        snn_NC = NC_noise(snn_NC, SRNN_args, STDP_lr)
        spk_rec_e1 = event_2_burst(spk_rec_e, SRNN_args)
        Spk_NC_tsp.append(torch.argwhere(spk_rec_e1 > 0))
    fr_NC = spk_rec_e1.detach().sum(dim=1)

    STDP_tuned = fr_STDP.detach()
    STDP_tuned[fr_BDP < 1.0 * fr_baseline] = 0.0
    STDP_tuned[fr_BP1 < 1.0 * fr_baseline] = 0.0
    STDP_tuned[fr_BP2 < 1.0 * fr_baseline] = 0.0
    tune_index0 = torch.argwhere(STDP_tuned >= 1.0 * fr_baseline)
    if tune_index0.size(0) < 2:
        continue

    for epoch in range(0, learning_epochs):
        spk0 = torch.zeros_like(spk_rec_e1)
        spk0[Spk_STDP_tsp[epoch][:,0], Spk_STDP_tsp[epoch][:,1], Spk_STDP_tsp[epoch][:,2]] = 1.0
        Spk_STDP_trace.append(spk0[tune_index0[:,0],:,tune_index0[:,1]].detach().cpu())
    Spk_STDP_trace = np.array(torch.concat(Spk_STDP_trace, dim=1))
    del spk0
    for epoch in range(0, learning_epochs):
        spk0 = torch.zeros_like(spk_rec_e1)
        spk0[Spk_BDP_tsp[epoch][:,0], Spk_BDP_tsp[epoch][:, 1], Spk_BDP_tsp[epoch][:, 2]] = 1.0
        Spk_BDP_trace.append(spk0[tune_index0[:,0],:,tune_index0[:,1]].detach().cpu())
    Spk_BDP_trace = np.array(torch.concat(Spk_BDP_trace, dim=1))
    del spk0
    for epoch in range(0, learning_epochs):
        spk0 = torch.zeros_like(spk_rec_e1)
        spk0[Spk_BP1_tsp[epoch][:,0], Spk_BP1_tsp[epoch][:,1], Spk_BP1_tsp[epoch][:,2]] = 1.0
        Spk_BP1_trace.append(spk0[tune_index0[:,0],:,tune_index0[:,1]].detach().cpu())
    Spk_BP1_trace = np.array(torch.concat(Spk_BP1_trace, dim=1))
    del spk0
    for epoch in range(0, learning_epochs):
        spk0 = torch.zeros_like(spk_rec_e1)
        spk0[Spk_BP2_tsp[epoch][:,0], Spk_BP2_tsp[epoch][:,1], Spk_BP2_tsp[epoch][:,2]] = 1.0
        Spk_BP2_trace.append(spk0[tune_index0[:,0],:,tune_index0[:,1]].detach().cpu())
    Spk_BP2_trace = np.array(torch.concat(Spk_BP2_trace, dim=1))
    del spk0
    for epoch in range(0, learning_epochs):
        spk0 = torch.zeros_like(spk_rec_e1)
        spk0[Spk_NC_tsp[epoch][:,0], Spk_NC_tsp[epoch][:,1], Spk_NC_tsp[epoch][:,2]] = 1.0
        Spk_NC_trace.append(spk0[tune_index0[:,0],:,tune_index0[:,1]].detach().cpu())
    Spk_NC_trace = np.array(torch.concat(Spk_NC_trace, dim=1))
    del spk0

    data_st = 500 * 0
    data_ed = 500 * 30
    SPK = np.concatenate((Spk_STDP_trace[:,data_st:data_ed],Spk_BDP_trace[:,data_st:data_ed],Spk_BP1_trace[:,data_st:data_ed],Spk_BP2_trace[:,data_st:data_ed],Spk_NC_trace[:,data_st:data_ed]),axis=0)
    ptype = np.zeros(len(tune_index0) * 5)
    ptype[0:len(tune_index0)] = 0
    ptype[len(tune_index0):2 * len(tune_index0)] = 1
    ptype[2 * len(tune_index0):3 * len(tune_index0)] = 2
    ptype[3 * len(tune_index0):4 * len(tune_index0)] = 3
    ptype[4 * len(tune_index0):5 * len(tune_index0)] = 4

    tune = tune_index0[:,1].unique()

    np.save(save_dir + f'/Spk_an_wn_{repeat_ind}.npy',SPK)
    np.save(save_dir + f'/Ptype_an_wn_{repeat_ind}.npy',ptype)
    np.save(save_dir + f'/Tune_{repeat_ind}.npy',tune.detach().cpu().numpy())

    np.save(save_dir + f'/snn_STDP_w1_{repeat_ind}.npy',snn_STDP.state_dict()['w1'].detach().cpu().numpy())
    np.save(save_dir + f'/snn_BDP_w1_{repeat_ind}.npy',snn_BDP.state_dict()['w1'].detach().cpu().numpy())
    np.save(save_dir + f'/snn_BP1_w1_{repeat_ind}.npy',snn_BP1.state_dict()['w1'].detach().cpu().numpy())
    np.save(save_dir + f'/snn_BP2_w1_{repeat_ind}.npy',snn_BP2.state_dict()['w1'].detach().cpu().numpy())
    np.save(save_dir + f'/snn_NC_w1_{repeat_ind}.npy',snn_NC.state_dict()['w1'].detach().cpu().numpy())

    repeat_ind += 1
    trace_num0 += np.size(Spk_STDP_trace, axis=0)
    print(f'trace_num: {trace_num0}')

<ipython-input-4-6c8474f5ba1d>:39: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  snn_BDP = torch.load(save_dir + '/snn_STDP.pth')
<ipython-input-4-6c8474f5ba1d>:50: FutureWa

trace_num: 4
trace_num: 22
trace_num: 50
trace_num: 63
trace_num: 71
trace_num: 81
trace_num: 90
trace_num: 117
trace_num: 131
trace_num: 138
trace_num: 163
trace_num: 183
trace_num: 204
trace_num: 226
trace_num: 244
trace_num: 264
trace_num: 268
trace_num: 276
trace_num: 283
trace_num: 285
trace_num: 308
trace_num: 325
trace_num: 329
trace_num: 343
trace_num: 353
trace_num: 369
trace_num: 376
trace_num: 379
trace_num: 393
trace_num: 415
trace_num: 450
trace_num: 453
trace_num: 455
trace_num: 464
trace_num: 468
trace_num: 476
trace_num: 481
trace_num: 495
trace_num: 505
trace_num: 533
trace_num: 538
trace_num: 546
trace_num: 561
trace_num: 563
trace_num: 565
trace_num: 568
trace_num: 590
trace_num: 595
trace_num: 610
